# Ch15: Inter-Agent（代理间通信）— LangChain + MiMo

本章演示 Agent 之间的通信模式：

1. **Agent Card**：定义 Agent 的能力和技能
2. **任务委托**：Agent A 将任务委托给 Agent B
3. **结果聚合**：汇总多个 Agent 的结果

## A2A 协议简介
A2A（Agent-to-Agent）是 Google 提出的代理间通信标准：
- 每个 Agent 通过 **Agent Card** 声明自己的能力
- 其他 Agent 可以通过 **Task** 向目标 Agent 发送请求
- 目标 Agent 执行任务并返回结果

## 与其他章节的关系
- Ch7 Multi-Agent：多 Agent 协作（内部编排）
- Ch10 MCP：Agent 与工具的通信协议
- Ch15 Inter-Agent：Agent 与 Agent 的通信协议（本章）

## 第1步：导入依赖

In [1]:
import os
import json
from typing import Optional, List, Dict, Any

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("MIMO_API_KEY", "")
os.environ["OPENAI_API_BASE"] = "https://token-plan-cn.xiaomimimo.com/v1"

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0.3,
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)
print("MiMo 模型初始化完成")

MiMo 模型初始化完成


## 第2步：定义 Agent Card

Agent Card 是 Agent 的「名片」，声明：
- 名称和描述
- 技能列表（能做什么）
- 认证方式

In [2]:
# Agent Card 定义

class AgentCard:
    """Agent Card：Agent 的能力声明"""
    def __init__(self, name: str, description: str, skills: List[Dict[str, str]]):
        self.name = name
        self.description = description
        self.skills = skills  # [{"name": "技能名", "description": "技能描述"}]

    def to_dict(self) -> dict:
        return {
            "name": self.name,
            "description": self.description,
            "skills": self.skills
        }

    def has_skill(self, skill_name: str) -> bool:
        return any(s["name"] == skill_name for s in self.skills)

    def __repr__(self):
        skills_str = ", ".join(s["name"] for s in self.skills)
        return f"AgentCard({self.name}, skills=[{skills_str}])"

# 定义两个 Agent
weather_agent_card = AgentCard(
    name="WeatherAgent",
    description="天气查询 Agent，可以查询任意城市的天气信息",
    skills=[
        {"name": "get_weather", "description": "查询指定城市的当前天气"},
        {"name": "get_forecast", "description": "查询指定城市未来3天天气预报"}
    ]
)

knowledge_agent_card = AgentCard(
    name="KnowledgeAgent",
    description="知识问答 Agent，可以回答各种知识问题",
    skills=[
        {"name": "answer_question", "description": "回答知识性问题"},
        {"name": "explain_concept", "description": "解释专业概念"}
    ]
)

print("Agent Cards 定义完成：")
print(f"  {weather_agent_card}")
print(f"  {knowledge_agent_card}")

Agent Cards 定义完成：
  AgentCard(WeatherAgent, skills=[get_weather, get_forecast])
  AgentCard(KnowledgeAgent, skills=[answer_question, explain_concept])


## 第3步：定义 Agent 和任务处理

每个 Agent 有自己的处理逻辑，可以处理来自其他 Agent 的任务请求。

In [3]:
# Agent 基类

class Agent:
    """Agent 基类"""
    def __init__(self, card: AgentCard, llm):
        self.card = card
        self.llm = llm

    def handle_task(self, task: dict) -> dict:
        """处理任务请求"""
        skill_name = task.get("skill")
        params = task.get("params", {})

        if not self.card.has_skill(skill_name):
            return {"error": f"Agent {self.card.name} 不支持技能: {skill_name}"}

        # 调用具体的处理方法
        method = getattr(self, skill_name, None)
        if method:
            return method(**params)
        else:
            return {"error": f"技能 {skill_name} 未实现"}

# WeatherAgent

class WeatherAgent(Agent):
    """天气查询 Agent"""

    def get_weather(self, city: str) -> dict:
        """查询天气（模拟）"""
        # 模拟天气数据
        weather_data = {
            "北京": {"temp": "22°C", "weather": "晴", "humidity": "45%"},
            "上海": {"temp": "25°C", "weather": "多云", "humidity": "60%"},
            "广州": {"temp": "30°C", "weather": "阵雨", "humidity": "80%"},
        }
        data = weather_data.get(city, {"temp": "20°C", "weather": "未知", "humidity": "50%"})
        return {"city": city, **data, "source": "WeatherAgent"}

    def get_forecast(self, city: str) -> dict:
        """查询天气预报（模拟）"""
        return {
            "city": city,
            "forecast": [
                {"day": "明天", "temp": "23°C", "weather": "晴"},
                {"day": "后天", "temp": "21°C", "weather": "多云"},
                {"day": "大后天", "temp": "19°C", "weather": "小雨"},
            ],
            "source": "WeatherAgent"
        }

# KnowledgeAgent

class KnowledgeAgent(Agent):
    """知识问答 Agent"""

    def answer_question(self, question: str) -> dict:
        """回答问题"""
        response = self.llm.invoke([HumanMessage(content=question)])
        return {"question": question, "answer": response.content, "source": "KnowledgeAgent"}

    def explain_concept(self, concept: str) -> dict:
        """解释概念"""
        prompt = f"请用简洁的语言解释以下概念：{concept}"
        response = self.llm.invoke([HumanMessage(content=prompt)])
        return {"concept": concept, "explanation": response.content, "source": "KnowledgeAgent"}

# 创建 Agent 实例
weather_agent = WeatherAgent(weather_agent_card, llm)
knowledge_agent = KnowledgeAgent(knowledge_agent_card, llm)

print("Agent 创建完成：")
print(f"  - {weather_agent.card.name}")
print(f"  - {knowledge_agent.card.name}")

Agent 创建完成：
  - WeatherAgent
  - KnowledgeAgent


## 第4步：Agent 间任务委托

演示 Agent A 如何发现 Agent B 的能力并委托任务。

In [4]:
# Agent 间任务委托

# 模拟 Agent 注册表
AGENT_REGISTRY = {
    "WeatherAgent": weather_agent,
    "KnowledgeAgent": knowledge_agent,
}

def delegate_task(target_agent_name: str, task: dict) -> dict:
    """向目标 Agent 委托任务"""
    agent = AGENT_REGISTRY.get(target_agent_name)
    if not agent:
        return {"error": f"未找到 Agent: {target_agent_name}"}

    print(f"  委托任务到 {target_agent_name}: skill={task.get('skill')}")
    result = agent.handle_task(task)
    return result

# 测试任务委托
print("=" * 60)
print("测试 Agent 间任务委托")
print("=" * 60)

# 任务1：查询天气
print("\n任务1：向 WeatherAgent 查询北京天气")
result1 = delegate_task("WeatherAgent", {"skill": "get_weather", "params": {"city": "北京"}})
print(f"  结果: {result1}")

# 任务2：解释概念
print("\n任务2：向 KnowledgeAgent 解释 RAG")
result2 = delegate_task("KnowledgeAgent", {"skill": "explain_concept", "params": {"concept": "RAG"}})
print(f"  结果: {result2['explanation'][:80]}...")

# 任务3：错误处理 - 不存在的技能
print("\n任务3：向 WeatherAgent 请求不存在的技能")
result3 = delegate_task("WeatherAgent", {"skill": "translate", "params": {"text": "hello"}})
print(f"  结果: {result3}")

测试 Agent 间任务委托

任务1：向 WeatherAgent 查询北京天气
  委托任务到 WeatherAgent: skill=get_weather
  结果: {'city': '北京', 'temp': '22°C', 'weather': '晴', 'humidity': '45%', 'source': 'WeatherAgent'}

任务2：向 KnowledgeAgent 解释 RAG
  委托任务到 KnowledgeAgent: skill=explain_concept


  结果: RAG（检索增强生成）是一种结合信息检索与文本生成的技术，通过从外部知识库中检索相关信息，再输入给大语言模型生成更准确、可靠的回答。...

任务3：向 WeatherAgent 请求不存在的技能
  委托任务到 WeatherAgent: skill=translate
  结果: {'error': 'Agent WeatherAgent 不支持技能: translate'}


## 第5步：协调者模式 — 多 Agent 协作

协调者（Coordinator）负责：
1. 理解用户需求
2. 决定调用哪些 Agent
3. 汇总结果返回给用户

In [5]:
# 协调者模式

COORDINATOR_PROMPT = """你是一个任务协调者。根据用户的需求，决定需要调用哪些 Agent。

可用的 Agent：
{agent_cards}

请返回 JSON 格式的任务计划：
{{
  "tasks": [
    {{"agent": "Agent名称", "skill": "技能名", "params": {{"参数名": "参数值"}}}}
  ]
}}

只返回 JSON，不要有其他内容。"""

def coordinator(user_request: str) -> dict:
    """协调者：分析需求 → 分配任务 → 汇总结果"""

    # 构建 Agent Card 信息
    cards_info = "\n".join([
        f"- {name}: {agent.card.description}\n  技能: {', '.join(s['name'] for s in agent.card.skills)}"
        for name, agent in AGENT_REGISTRY.items()
    ])

    # 让 LLM 决定任务分配
    prompt = COORDINATOR_PROMPT.format(agent_cards=cards_info)
    response = llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=user_request)
    ])

    # 解析任务计划
    text = response.content.strip()
    try:
        if "```json" in text:
            text = text.split("```json")[1].split("```")[0].strip()
        elif "```" in text:
            text = text.split("```")[1].split("```")[0].strip()
        plan = json.loads(text)
    except json.JSONDecodeError:
        return {"error": "无法解析任务计划", "raw": response.content}

    # 执行任务
    results = []
    for task in plan.get("tasks", []):
        agent_name = task.get("agent")
        result = delegate_task(agent_name, task)
        results.append(result)

    return {
        "user_request": user_request,
        "plan": plan,
        "results": results
    }

print("协调者定义完成")

协调者定义完成


In [6]:
# 测试协调者模式

print("=" * 60)
print("测试协调者模式")
print("=" * 60)

# 测试1：单 Agent 任务
print("\n--- 测试1：单 Agent 任务 ---")
result = coordinator("北京今天天气怎么样？")
print(f"用户请求: {result['user_request']}")
print(f"任务计划: {result['plan']}")
for r in result['results']:
    print(f"结果: {r}")

# 测试2：多 Agent 协作
print("\n--- 测试2：多 Agent 协作 ---")
result = coordinator("请告诉我上海的天气，顺便解释一下什么是 RAG")
print(f"用户请求: {result['user_request']}")
print(f"任务计划: {result['plan']}")
for r in result['results']:
    source = r.get('source', 'unknown')
    if 'answer' in r:
        print(f"[{source}] 回答: {r['answer'][:80]}...")
    elif 'explanation' in r:
        print(f"[{source}] 解释: {r['explanation'][:80]}...")
    elif 'city' in r:
        print(f"[{source}] 天气: {r}")

测试协调者模式

--- 测试1：单 Agent 任务 ---


  委托任务到 WeatherAgent: skill=get_weather
用户请求: 北京今天天气怎么样？
任务计划: {'tasks': [{'agent': 'WeatherAgent', 'skill': 'get_weather', 'params': {'city': '北京'}}]}
结果: {'city': '北京', 'temp': '22°C', 'weather': '晴', 'humidity': '45%', 'source': 'WeatherAgent'}

--- 测试2：多 Agent 协作 ---


  委托任务到 WeatherAgent: skill=get_weather
  委托任务到 KnowledgeAgent: skill=explain_concept


用户请求: 请告诉我上海的天气，顺便解释一下什么是 RAG
任务计划: {'tasks': [{'agent': 'WeatherAgent', 'skill': 'get_weather', 'params': {'city': '上海'}}, {'agent': 'KnowledgeAgent', 'skill': 'explain_concept', 'params': {'concept': 'RAG'}}]}
[WeatherAgent] 天气: {'city': '上海', 'temp': '25°C', 'weather': '多云', 'humidity': '60%', 'source': 'WeatherAgent'}
[KnowledgeAgent] 解释: RAG（检索增强生成）是一种结合信息检索与文本生成的技术。它先从外部知识库中检索相关文档，再将检索结果作为上下文输入给语言模型，从而生成更准确、有依据的回答。简...


## 总结

### A2A 通信模式

```
Agent Card（能力声明）
    ↓
Agent Registry（注册表）
    ↓
Coordinator（协调者）→ 任务分配 → Agent A / Agent B
    ↓
结果聚合 → 返回用户
```

### 核心要点
- **Agent Card**：声明能力，让其他 Agent 知道你能做什么
- **任务委托**：通过注册表发现目标 Agent，发送任务请求
- **协调者模式**：一个主 Agent 协调多个专业 Agent

### 三种通信模式对比

| 模式 | 说明 | 复杂度 | 适用场景 |
|------|------|--------|----------|
| 直接调用 | Agent A 直接调用 Agent B | 低 | 简单链式流程 |
| 注册表 | 通过注册表发现 Agent | 中 | 动态 Agent 发现 |
| 协调者 | 主 Agent 统一调度 | 高 | 复杂多 Agent 系统 |

### 与其他协议的关系

| 协议 | 通信对象 | 章节 |
|------|---------|------|
| Tool Use | Agent ↔ 工具 | Ch5 |
| MCP | Agent ↔ 服务 | Ch10 |
| A2A | Agent ↔ Agent | Ch15（本章）|

### 实际应用
- 多 Agent 客服系统（路由 Agent + 专业 Agent）
- 自动化工作流（编排 Agent + 执行 Agent）
- 分布式 Agent 网络（跨服务的 Agent 协作）